# Incremental mapping of climate change risks

notebook `climate_change` ,:
- `H0` Risk level chart (Lambert main picture + South China Sea small picture)
- `H1` Risk level chart (Lambert main picture + South China Sea small picture)
- Change rate distribution diagram of `H0-H1` relative to the `H0` baseline (Lambert main picture + South China Sea small picture)
- corresponds to GeoTIFF
- Provincial original GWR risk change rate lollipop chart


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import geopandas as gpd
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import numpy as np
import pandas as pd
import rasterio
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, TwoSlopeNorm
from pyproj import Transformer
from rasterio.transform import from_origin
from scipy.interpolate import griddata
from shapely.geometry import Point

import cartopy.crs as ccrs
import cartopy.feature as cfeature

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.serif"] = ["Times New Roman"]
plt.rcParams["font.size"] = 18
plt.rcParams["svg.fonttype"] = "none"


In [ ]:
ssp = "ssp2"  # TODO: 'hist' / 'ssp1' / 'ssp2' / 'ssp3' / 'ssp4' / 'ssp5'
ssp_time = "2080"  # TODO: '2020' / '2040' / '2060' / '2080' / '2100'

## 1. Parameter and path settings

This continues the usage of the previous notebook, and all paths are defined directly in the file.
This notebook will no longer switch between `metric`; the risk increment is fixedly defined as `H0-H1`, the third map is fixed to draw the change rate distribution relative to the `H0` baseline, and the level map is fixed to use sigmoid + saved natural breakpoint classification results.


In [ ]:
# =========================
# Manual parameter area
# =========================
base_path = "/path/to/sinkhole-risk-china"
resolution = "10km"

# If not specified separately, the raster resolution will be automatically derived based on resolution
resolution_km = None

# If you want to draw other existing CSV, you can manually specify it here; otherwise, the default file in the climate_change directory will be automatically read.
point_csv = None
province_csv = None

# =========================
# Processing step.
# =========================
path_name = f"Points_China_all_{resolution}"
out_dir = os.path.join(base_path, "outputs", path_name, ssp, ssp_time)
climate_dir = os.path.join(out_dir, "climate_change")

if point_csv is None:
    point_csv = os.path.join(
        climate_dir,
        f"climate_change_point_results_{path_name}_{ssp}_{ssp_time}.csv",
    )

if province_csv is None:
    province_csv = os.path.join(
        climate_dir,
        f"province_climate_change_summary_{path_name}_{ssp}_{ssp_time}.csv",
    )

os.makedirs(climate_dir, exist_ok=True)

print(f"point_csv: {point_csv}")
print(f"province_csv: {province_csv}")
print(f"climate_dir: {climate_dir}")


## 2. , 

This part is responsible for:
- Implementation note.
- Implementation note.
- Reuse the boundary, scale and rasterization logic of the original `run_graph.ipynb`


In [ ]:
COLORS_BY_TIME = {
    "2020": ["#e1e1e1", "#c5bcbd", "#aa989a", "#8e7376", "#724e52"],
    "2040": ["#e1e1e1", "#c5bcbd", "#aa989a", "#8e7376", "#724e52"],
    "2060": ["#e1e1e1", "#c5bcbd", "#aa989a", "#8e7376", "#724e52"],
    "2080": ["#e1e1e1", "#c5bcbd", "#aa989a", "#8e7376", "#724e52"],
    "2100": ["#e1e1e1", "#c5bcbd", "#aa989a", "#8e7376", "#724e52"],
}

class_names = {0: "Lowest", 1: "Low", 2: "Middle", 3: "High", 4: "Highest"}

try:
    colors = COLORS_BY_TIME[ssp_time]
except KeyError:
    raise ValueError(f"Unknown ssp_time:{ssp_time}, optional values:{list(COLORS_BY_TIME.keys())}")

print("ssp_time =", ssp_time)
print("colors =", colors)


def infer_resolution_km(resolution: str, override=None) -> float:
    if override is not None:
        return float(override)
    return float(str(resolution).lower().replace("km", ""))


def load_result_tables(point_csv: str, province_csv: str):
    if not os.path.exists(point_csv):
        raise FileNotFoundError(f"Point result CSV not found: {point_csv}")
    if not os.path.exists(province_csv):
        raise FileNotFoundError(f"Province summary CSV not found: {province_csv}")
    return (
        pd.read_csv(point_csv, encoding="utf-8-sig"),
        pd.read_csv(province_csv, encoding="utf-8-sig"),
    )


def load_boundary_data(base_path: str):
    """Load border data: provincial borders (excluding Hong Kong, Macao and Taiwan) + Hong Kong, Macao and Taiwan provincial borders (separate) + world borders + nine-dash line."""
    province_no_TW_AM_HK_shp = os.path.join(
        base_path,
        "data",
        "Administrative_divisions_of_china",
        "no_TW_AM_HK",
        "china_pro2_no_TW_AM_HK.shp",
    )
    province_no_TW_AM_HK = gpd.read_file(province_no_TW_AM_HK_shp)

    province_TW_AM_HK_shp = os.path.join(
        base_path,
        "data",
        "Administrative_divisions_of_china",
        "TW_AM_HK",
        "TW_AM_HK.shp",
    )
    province_TW_AM_HK = gpd.read_file(province_TW_AM_HK_shp)

    world_shp = os.path.join(base_path, "data", "world_SHP", "World_countries.shp")
    world_boundary = gpd.read_file(world_shp)

    nine_dash_line_path = os.path.join(base_path, "data", "china_SHP", "nanhai", "nine-dash line.shp")
    nine_dash_line = gpd.read_file(nine_dash_line_path)

    return province_no_TW_AM_HK, province_TW_AM_HK, world_boundary, nine_dash_line


def create_masked_raster(
    pre_df: pd.DataFrame,
    value_col: str,
    boundary_shp: str,
    output_path: str,
    resolution_km: float,
    nodata,
    dtype: str,
):
    """Strictly follow the original notebook’s mapping rasterization idea, and only change the value column to be configurable."""
    print(f"Start creating raster, fields:{value_col}, resolution:{resolution_km}km...")

    china_boundary = gpd.read_file(boundary_shp).to_crs("EPSG:3857")
    bounds = china_boundary.total_bounds
    expansion_factor_degree = 5
    expansion_m = expansion_factor_degree * 111000
    bounds[0] -= expansion_m
    bounds[1] -= expansion_m
    bounds[2] += expansion_m
    bounds[3] += expansion_m

    resolution_m = resolution_km * 1000.0
    x_min, y_min, x_max, y_max = bounds
    width = x_max - x_min
    height = y_max - y_min
    x_res = int(np.round(width / resolution_m))
    y_res = int(np.round(height / resolution_m))
    x_max_adj = x_min + x_res * resolution_m
    y_max_adj = y_min + y_res * resolution_m
    x = np.linspace(x_min, x_max_adj, x_res)
    y = np.linspace(y_min, y_max_adj, y_res)
    xx, yy = np.meshgrid(x, y)

    geometry = [Point(lon, lat) for lon, lat in zip(pre_df["Longitude"], pre_df["Latitude"])]
    gdf_points = gpd.GeoDataFrame(pre_df, geometry=geometry, crs="EPSG:4326").to_crs("EPSG:3857")
    points_coords = np.array([(pt.x, pt.y) for pt in gdf_points.geometry])
    values = pd.to_numeric(pre_df[value_col], errors="coerce").to_numpy(dtype=float)

    valid = np.isfinite(values)
    if not valid.any():
        raise ValueError(f"{value_col}No valid value for interpolation.")

    grid_values = griddata(
        points_coords[valid],
        values[valid],
        (xx, yy),
        method="nearest",
        fill_value=nodata,
    )

    grid_points = [Point(xv, yv) for xv, yv in zip(xx.ravel(), yy.ravel())]
    grid_gdf = gpd.GeoDataFrame(geometry=grid_points, crs="EPSG:3857")
    points_in_china = gpd.sjoin(grid_gdf, china_boundary, how="inner", predicate="within")
    mask = np.zeros(xx.shape, dtype=bool)
    mask.ravel()[points_in_china.index] = True
    grid_values[~mask] = nodata

    transform = from_origin(x_min, y_max_adj, resolution_m, resolution_m)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    if dtype == "uint8":
        out_arr = np.asarray(np.rint(grid_values), dtype=np.uint8)
        raster_dtype = "uint8"
    else:
        out_arr = np.asarray(grid_values, dtype=np.float32)
        raster_dtype = "float32"

    with rasterio.open(
        output_path,
        "w",
        driver="GTiff",
        height=out_arr.shape[0],
        width=out_arr.shape[1],
        count=1,
        dtype=raster_dtype,
        crs=china_boundary.crs,
        transform=transform,
        nodata=nodata,
    ) as dst:
        dst.write(out_arr, 1)

    print(f"TIF file saved:{output_path}")
    return out_arr, xx, yy, (x_min, y_min, x_max_adj, y_max_adj), transform


def add_scalebar(ax, lon0, lat0, length, projection, bar_lw=5, outline_lw=0.5):
    """Strictly follow the scale drawing method of the original notebook."""
    x0, y0 = projection.transform_point(lon0, lat0, ccrs.PlateCarree())
    scale_length = length * 1000
    outline_effect = [
        pe.Stroke(linewidth=bar_lw + 2 * outline_lw, foreground="black"),
        pe.Normal(),
    ]

    ax.plot(
        [x0, x0 + scale_length / 2],
        [y0, y0],
        color="black",
        lw=bar_lw,
        solid_capstyle="butt",
        transform=ax.transData,
        path_effects=outline_effect,
        zorder=5,
        clip_on=False,
    )
    ax.plot(
        [x0 + scale_length / 2, x0 + scale_length],
        [y0, y0],
        color="white",
        lw=bar_lw,
        solid_capstyle="butt",
        transform=ax.transData,
        path_effects=outline_effect,
        zorder=5,
        clip_on=False,
    )

    half_thick_pt = bar_lw / 2.0 + outline_lw
    half_thick_px = ax.figure.dpi * half_thick_pt / 72.0
    x_disp, y_disp = ax.transData.transform((x0, y0))
    _, y_up = ax.transData.inverted().transform((x_disp, y_disp + half_thick_px))
    _, y_dn = ax.transData.inverted().transform((x_disp, y_disp - half_thick_px))

    for x_end in (x0, x0 + scale_length):
        ax.plot(
            [x_end, x_end],
            [y_dn, y_up],
            color="black",
            lw=2 * outline_lw,
            solid_capstyle="butt",
            transform=ax.transData,
            zorder=6,
            clip_on=False,
        )

    label_y = y0 + scale_length * 0.028
    ax.text(x0, label_y, "0", ha="center", va="bottom", fontfamily="Times New Roman", fontsize=30.0, transform=ax.transData, zorder=7)
    ax.text(x0 + scale_length / 2, label_y, "500", ha="center", va="bottom", fontfamily="Times New Roman", fontsize=30.0, transform=ax.transData, zorder=7)
    ax.text(x0 + scale_length + scale_length * 0.065, label_y, f"{length} km", ha="left", va="bottom", fontfamily="Times New Roman", fontsize=30.0, transform=ax.transData, zorder=7)


## 3. Map and provincial change rate drawing function

This part is responsible for:
- susceptibility map Lambert
- Strictly reuse the original Nanhai small drawing method
- Implementation note.
- Change the third picture to the continuous change rate distribution of `((H0-H1)/|H0|) * 100`, and keep the rest of the layout consistent.


In [ ]:
def style_geo_frame(ax, edgecolor="#4a4a4a", linewidth=0.9):
    if hasattr(ax, "outline_patch") and ax.outline_patch is not None:
        ax.outline_patch.set_edgecolor(edgecolor)
        ax.outline_patch.set_linewidth(linewidth)
        ax.outline_patch.set_zorder(12)
    if "geo" in ax.spines:
        ax.spines["geo"].set_edgecolor(edgecolor)
        ax.spines["geo"].set_linewidth(linewidth)
        ax.spines["geo"].set_zorder(12)


def add_axis_titles(fig, xlabel=None, ylabel=None, xlabel_y=0.043, ylabel_x=0.032, fontsize=20):
    if xlabel:
        fig.text(0.50, xlabel_y, xlabel, ha="center", va="center", fontsize=fontsize, family="Times New Roman")
    if ylabel:
        fig.text(ylabel_x, 0.52, ylabel, ha="center", va="center", rotation=90, fontsize=fontsize, family="Times New Roman")


def add_nanhai_inset_axes(fig, width_ratio=1.90 / 12.0, height_ratio=6.0 / 8.0, right=0.902, bottom=0.171):
    inset_width = width_ratio
    inset_height = inset_width * (fig.get_figwidth() / fig.get_figheight()) * height_ratio
    inset_left = right - inset_width
    return fig.add_axes([inset_left, bottom, inset_width, inset_height], projection=ccrs.PlateCarree())


def draw_nanhai_inset_categorical(ax, grid_values, bounds, province_no_TW_AM_HK, province_TW_AM_HK, world_boundary, nine_dash_line):
    province_boundary = gpd.GeoDataFrame(pd.concat([province_no_TW_AM_HK, province_TW_AM_HK], ignore_index=True), crs=province_no_TW_AM_HK.crs)
    cmap = ListedColormap(colors)
    proj = ccrs.PlateCarree()
    ax.set_extent([105, 120, 2, 24], crs=proj)
    ax.add_feature(cfeature.LAND, color="#F7F7F7", alpha=0.8)
    ax.add_feature(cfeature.OCEAN, color="#CECECE", alpha=0.8)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.8)

    try:
        proj4_str = proj.proj4_init
    except AttributeError:
        proj4_str = proj.as_proj4()

    world_boundary_proj = world_boundary.to_crs(proj4_str)
    ax.add_geometries(world_boundary_proj.geometry, crs=proj, facecolor="none", edgecolor="gray", linewidth=0.3, alpha=0.6)
    province_boundary_proj = province_boundary.to_crs(proj4_str)
    ax.add_geometries(province_boundary_proj.geometry, crs=proj, facecolor="none", edgecolor="black", linewidth=0.5, alpha=0.8)
    nine_dash_line_proj = nine_dash_line.to_crs(proj4_str)
    ax.add_geometries(nine_dash_line_proj.geometry, crs=proj, facecolor="none", edgecolor="black", linewidth=1.5, alpha=1.0, linestyle="--")

    data_to_plot = np.ma.masked_where(grid_values == 255, grid_values)
    x_mercator = np.linspace(bounds[0], bounds[2], data_to_plot.shape[1])
    y_mercator = np.linspace(bounds[1], bounds[3], data_to_plot.shape[0])
    xx_mercator, yy_mercator = np.meshgrid(x_mercator, y_mercator)
    transformer_to_wgs84 = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
    xx_wgs84, yy_wgs84 = transformer_to_wgs84.transform(xx_mercator, yy_mercator)
    ax.pcolormesh(xx_wgs84, yy_wgs84, data_to_plot, cmap=cmap, vmin=0, vmax=4, alpha=0.7, transform=proj)

    ax.set_xticks([])
    ax.set_yticks([])
    style_geo_frame(ax, edgecolor="#8a8a8a", linewidth=0.8)
    ax.set_facecolor("white")


def create_preview_map_categorical(grid_values, bounds, output_base_path, resolution_km, base_path):
    """Strictly reuse the Lambert drawing method of the original susceptibility map, and embed the small map of the South China Sea in the lower right corner."""
    print(f": Nature ,:{resolution_km}km")

    province_no_TW_AM_HK, province_TW_AM_HK, world_boundary, nine_dash_line = load_boundary_data(base_path)
    cmap = ListedColormap(colors)

    fig = plt.figure(figsize=(12, 10))
    proj = ccrs.LambertConformal(central_longitude=105, standard_parallels=(25, 47))
    ax = fig.add_axes([0.065, 0.14, 0.87, 0.79], projection=proj)
    ax.set_extent([80, 128, 18, 54], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, color="#fefefe", alpha=0.8)
    ax.add_feature(cfeature.OCEAN, color="#bebebe", alpha=0.8)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.6)

    try:
        proj4_str = proj.proj4_init
    except AttributeError:
        proj4_str = proj.as_proj4()

    data_to_plot = np.ma.masked_where(grid_values == 255, grid_values)
    x_mercator = np.linspace(bounds[0], bounds[2], data_to_plot.shape[1])
    y_mercator = np.linspace(bounds[1], bounds[3], data_to_plot.shape[0])
    xx_mercator, yy_mercator = np.meshgrid(x_mercator, y_mercator)
    transformer_to_lambert = Transformer.from_crs("EPSG:3857", proj4_str, always_xy=True)
    xx_lambert, yy_lambert = transformer_to_lambert.transform(xx_mercator, yy_mercator)
    x_min_lambert, y_min_lambert = transformer_to_lambert.transform(bounds[0], bounds[1])
    x_max_lambert, y_max_lambert = transformer_to_lambert.transform(bounds[2], bounds[3])
    x_lambert_regular = np.linspace(x_min_lambert, x_max_lambert, data_to_plot.shape[1])
    y_lambert_regular = np.linspace(y_min_lambert, y_max_lambert, data_to_plot.shape[0])
    points_lambert = np.array([xx_lambert.ravel(), yy_lambert.ravel()]).T
    values_lambert = data_to_plot.ravel()
    xx_lambert_reg, yy_lambert_reg = np.meshgrid(x_lambert_regular, y_lambert_regular)
    data_lambert_reg = griddata(points_lambert, values_lambert, (xx_lambert_reg, yy_lambert_reg), method="nearest")
    data_lambert_reg = np.ma.masked_where(data_lambert_reg == 255, data_lambert_reg)
    extent = [x_lambert_regular.min(), x_lambert_regular.max(), y_lambert_regular.min(), y_lambert_regular.max()]

    ax.imshow(data_lambert_reg, cmap=cmap, vmin=0, vmax=4, extent=extent, origin="lower", alpha=0.8, transform=proj, interpolation="nearest", zorder=2)

    province_no_proj = province_no_TW_AM_HK.to_crs(proj4_str)
    ax.add_geometries(province_no_proj.geometry, crs=proj, facecolor="none", edgecolor="black", linewidth=0.25, alpha=0.8, zorder=4)
    province_tw_proj = province_TW_AM_HK.to_crs(proj4_str)
    ax.add_geometries(province_tw_proj.geometry, crs=proj, facecolor="white", edgecolor="black", linewidth=0.25, alpha=1.0, zorder=5)
    china_outline_geom = province_no_proj.unary_union.union(province_tw_proj.unary_union)
    ax.add_geometries([china_outline_geom], crs=proj, facecolor="none", edgecolor="black", linewidth=0.8, alpha=1.0, zorder=6)
    nine_dash_line_proj = nine_dash_line.to_crs(proj4_str)
    ax.add_geometries(nine_dash_line_proj.geometry, crs=proj, facecolor="none", edgecolor="black", linewidth=0.6, alpha=0.9, zorder=6)

    lon_ticks = [80, 90, 100, 110, 120, 130]
    lat_ticks = [20, 30, 40, 50]
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, linestyle="--", alpha=0.3, color="gray", rotate_labels=False)
    gl.xlocator = plt.FixedLocator(lon_ticks)
    gl.ylocator = plt.FixedLocator(lat_ticks)
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = True
    gl.bottom_labels = True
    gl.xlabel_style = {"size": 32, "family": "Times New Roman", "rotation": 0}
    gl.ylabel_style = {"size": 32, "family": "Times New Roman", "rotation": 0}
    gl.xlines = False
    gl.ylines = False
    gl.xformatter = LongitudeFormatter()
    gl.yformatter = LatitudeFormatter()
    style_geo_frame(ax)

    legend_elements = [plt.Rectangle((0, 0), 1, 1, facecolor="white", edgecolor="black", linewidth=1, label="No data")]
    for i in range(5):
        legend_elements.append(plt.Rectangle((0, 0), 1, 1, facecolor=colors[i], edgecolor="black", linewidth=1, label=class_names[i]))

    legend = ax.legend(handles=legend_elements, loc="lower left", bbox_to_anchor=(-0.005, 0.005), frameon=True, fancybox=False, framealpha=0, edgecolor="none", ncol=2, fontsize=26, handlelength=1.8, handleheight=0.9, columnspacing=0.95, borderpad=0.2, labelspacing=0.35)
    legend.set_title("Susceptibility Level", prop={"family": "Times New Roman", "size": 28, "weight": "normal"})
    legend.get_frame().set_facecolor("none")
    legend.get_frame().set_edgecolor("none")
    legend.get_title().set_fontfamily("Times New Roman")

    add_scalebar(ax, 70, 49, 1000, proj)
    add_axis_titles(fig, xlabel="Longitude", ylabel="Latitude")
    inset_ax = add_nanhai_inset_axes(fig, right=0.9384, bottom=0.1423)
    draw_nanhai_inset_categorical(inset_ax, grid_values, bounds, province_no_TW_AM_HK, province_TW_AM_HK, world_boundary, nine_dash_line)

    output_png = f"{output_base_path}.png"
    fig.canvas.draw()
    plt.savefig(output_png, bbox_inches="tight", pad_inches=1 / 25.4, facecolor="white", edgecolor="none", dpi=600, format="png")
    print(f"PNG:{output_png}")
    plt.show()
    return fig, ax


def build_increment_norm(grid_values: np.ndarray) -> TwoSlopeNorm:
    """H0-H1 -50% 50%."""
    return TwoSlopeNorm(vmin=-50.0, vcenter=0.0, vmax=50.0)


def draw_nanhai_inset_continuous(ax, grid_values, bounds, province_no_TW_AM_HK, province_TW_AM_HK, world_boundary, nine_dash_line):
    province_boundary = gpd.GeoDataFrame(pd.concat([province_no_TW_AM_HK, province_TW_AM_HK], ignore_index=True), crs=province_no_TW_AM_HK.crs)
    cmap = LinearSegmentedColormap.from_list("climate_increment_raw", [(0.00, "#4f7f93"), (0.25, "#a9c8d4"), (0.50, "#f7f4ef"), (0.75, "#dfbeb4"), (1.00, "#a55d4b")])
    norm = build_increment_norm(grid_values)
    proj = ccrs.PlateCarree()
    ax.set_extent([105, 120, 2, 24], crs=proj)
    ax.add_feature(cfeature.LAND, color="#F7F7F7", alpha=0.8)
    ax.add_feature(cfeature.OCEAN, color="#CECECE", alpha=0.8)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.8)

    try:
        proj4_str = proj.proj4_init
    except AttributeError:
        proj4_str = proj.as_proj4()

    world_boundary_proj = world_boundary.to_crs(proj4_str)
    ax.add_geometries(world_boundary_proj.geometry, crs=proj, facecolor="none", edgecolor="gray", linewidth=0.3, alpha=0.6)
    province_boundary_proj = province_boundary.to_crs(proj4_str)
    ax.add_geometries(province_boundary_proj.geometry, crs=proj, facecolor="none", edgecolor="black", linewidth=0.5, alpha=0.8)
    nine_dash_line_proj = nine_dash_line.to_crs(proj4_str)
    ax.add_geometries(nine_dash_line_proj.geometry, crs=proj, facecolor="none", edgecolor="black", linewidth=1.5, alpha=1.0, linestyle="--")

    data_to_plot = np.ma.masked_where(grid_values == -9999, grid_values)
    x_mercator = np.linspace(bounds[0], bounds[2], data_to_plot.shape[1])
    y_mercator = np.linspace(bounds[1], bounds[3], data_to_plot.shape[0])
    xx_mercator, yy_mercator = np.meshgrid(x_mercator, y_mercator)
    transformer_to_wgs84 = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
    xx_wgs84, yy_wgs84 = transformer_to_wgs84.transform(xx_mercator, yy_mercator)
    ax.pcolormesh(xx_wgs84, yy_wgs84, data_to_plot, cmap=cmap, norm=norm, alpha=0.82, transform=proj)

    ax.set_xticks([])
    ax.set_yticks([])
    style_geo_frame(ax, edgecolor="#8a8a8a", linewidth=0.8)
    ax.set_facecolor("white")


def create_preview_map_continuous(grid_values, bounds, output_base_path, resolution_km, base_path, colorbar_label):
    """Keep the original map base map and layout, only change the coloring to the original GWR incremental continuous value, and embed the small map of the South China Sea in the lower right corner."""
    print(f": GWR Lambert ,:{resolution_km}km")

    province_no_TW_AM_HK, province_TW_AM_HK, world_boundary, nine_dash_line = load_boundary_data(base_path)
    cmap = LinearSegmentedColormap.from_list("climate_increment_raw", [(0.00, "#4f7f93"), (0.25, "#a9c8d4"), (0.50, "#f7f4ef"), (0.75, "#dfbeb4"), (1.00, "#a55d4b")])
    norm = build_increment_norm(grid_values)

    fig = plt.figure(figsize=(12, 10))
    proj = ccrs.LambertConformal(central_longitude=105, standard_parallels=(25, 47))
    ax = fig.add_axes([0.065, 0.17, 0.87, 0.76], projection=proj)
    ax.set_extent([80, 128, 18, 54], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, color="#fefefe", alpha=0.8)
    ax.add_feature(cfeature.OCEAN, color="#bebebe", alpha=0.8)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.6)

    try:
        proj4_str = proj.proj4_init
    except AttributeError:
        proj4_str = proj.as_proj4()

    data_to_plot = np.ma.masked_where(grid_values == -9999, grid_values)
    x_mercator = np.linspace(bounds[0], bounds[2], data_to_plot.shape[1])
    y_mercator = np.linspace(bounds[1], bounds[3], data_to_plot.shape[0])
    xx_mercator, yy_mercator = np.meshgrid(x_mercator, y_mercator)
    transformer_to_lambert = Transformer.from_crs("EPSG:3857", proj4_str, always_xy=True)
    xx_lambert, yy_lambert = transformer_to_lambert.transform(xx_mercator, yy_mercator)
    x_min_lambert, y_min_lambert = transformer_to_lambert.transform(bounds[0], bounds[1])
    x_max_lambert, y_max_lambert = transformer_to_lambert.transform(bounds[2], bounds[3])
    x_lambert_regular = np.linspace(x_min_lambert, x_max_lambert, data_to_plot.shape[1])
    y_lambert_regular = np.linspace(y_min_lambert, y_max_lambert, data_to_plot.shape[0])
    points_lambert = np.array([xx_lambert.ravel(), yy_lambert.ravel()]).T
    values_lambert = data_to_plot.ravel()
    xx_lambert_reg, yy_lambert_reg = np.meshgrid(x_lambert_regular, y_lambert_regular)
    data_lambert_reg = griddata(points_lambert, values_lambert, (xx_lambert_reg, yy_lambert_reg), method="nearest")
    data_lambert_reg = np.ma.masked_where(data_lambert_reg == -9999, data_lambert_reg)
    extent = [x_lambert_regular.min(), x_lambert_regular.max(), y_lambert_regular.min(), y_lambert_regular.max()]

    im = ax.imshow(data_lambert_reg, cmap=cmap, norm=norm, extent=extent, origin="lower", alpha=0.88, transform=proj, interpolation="nearest", zorder=2)
    province_no_proj = province_no_TW_AM_HK.to_crs(proj4_str)
    ax.add_geometries(province_no_proj.geometry, crs=proj, facecolor="none", edgecolor="black", linewidth=0.25, alpha=0.8, zorder=4)
    province_tw_proj = province_TW_AM_HK.to_crs(proj4_str)
    ax.add_geometries(province_tw_proj.geometry, crs=proj, facecolor="white", edgecolor="black", linewidth=0.25, alpha=1.0, zorder=5)
    china_outline_geom = province_no_proj.unary_union.union(province_tw_proj.unary_union)
    ax.add_geometries([china_outline_geom], crs=proj, facecolor="none", edgecolor="black", linewidth=0.8, alpha=1.0, zorder=6)
    nine_dash_line_proj = nine_dash_line.to_crs(proj4_str)
    ax.add_geometries(nine_dash_line_proj.geometry, crs=proj, facecolor="none", edgecolor="black", linewidth=0.6, alpha=0.9, zorder=6)

    lon_ticks = [80, 90, 100, 110, 120, 130]
    lat_ticks = [20, 30, 40, 50]
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, linestyle="--", alpha=0.3, color="gray", rotate_labels=False)
    gl.xlocator = plt.FixedLocator(lon_ticks)
    gl.ylocator = plt.FixedLocator(lat_ticks)
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = True
    gl.bottom_labels = True
    gl.xlabel_style = {"size": 32, "family": "Times New Roman", "rotation": 0}
    gl.ylabel_style = {"size": 32, "family": "Times New Roman", "rotation": 0}
    gl.xlines = False
    gl.ylines = False
    gl.xformatter = LongitudeFormatter()
    gl.yformatter = LatitudeFormatter()
    style_geo_frame(ax)

    cax = fig.add_axes([0.26, 0.075, 0.44, 0.028])
    cbar = fig.colorbar(im, cax=cax, orientation="horizontal")
    cbar.ax.tick_params(labelsize=32)

    add_scalebar(ax, 70, 49, 1000, proj)
    inset_ax = add_nanhai_inset_axes(fig, right=0.926, bottom=0.1729)
    draw_nanhai_inset_continuous(inset_ax, grid_values, bounds, province_no_TW_AM_HK, province_TW_AM_HK, world_boundary, nine_dash_line)
    output_png = f"{output_base_path}.png"
    fig.canvas.draw()
    plt.savefig(output_png, bbox_inches="tight", pad_inches=1 / 25.4, facecolor="white", edgecolor="none", dpi=600, format="png")
    print(f"PNG:{output_png}")
    plt.show()
    return fig, ax


def build_point_colors(values: np.ndarray):
    pos_cmap = LinearSegmentedColormap.from_list("rise", ["#d9d1d3", "#8b666f"])
    neg_cmap = LinearSegmentedColormap.from_list("decline", ["#d9eaee", "#77bed0"])

    values = np.asarray(values, dtype=float)
    pos_values = values[values > 0]
    neg_values = np.abs(values[values < 0])

    pos_min = pos_values.min() if pos_values.size else 0.0
    pos_max = pos_values.max() if pos_values.size else 1.0
    neg_min = neg_values.min() if neg_values.size else 0.0
    neg_max = neg_values.max() if neg_values.size else 1.0

    colors_local = []
    for value in values:
        if value > 0:
            if pos_max - pos_min < 1e-12:
                t = 1.0
            else:
                t = (value - pos_min) / (pos_max - pos_min)
            colors_local.append(pos_cmap(0.25 + 0.70 * t))
        elif value < 0:
            magnitude = abs(value)
            if neg_max - neg_min < 1e-12:
                t = 1.0
            else:
                t = (magnitude - neg_min) / (neg_max - neg_min)
            colors_local.append(neg_cmap(0.25 + 0.70 * t))
        else:
            colors_local.append("#d9dfe2")
    return colors_local


def draw_lollipops(ax, x, y, colors_local, fig, stem_width=3.6, marker_size=235):
    highlight_offset = mtransforms.ScaledTranslation(-2.2 / 72.0, 2.2 / 72.0, fig.dpi_scale_trans)
    for xi, yi, ci in zip(x, y, colors_local):
        ax.vlines(xi, 0, yi, color=ci, linewidth=stem_width, zorder=2, alpha=0.98)
        ax.scatter(xi, yi, s=marker_size, color=[ci], edgecolor=[ci], linewidth=0.8, zorder=4)
        ax.scatter(
            [xi],
            [yi],
            s=36,
            color="white",
            alpha=0.82,
            edgecolor="none",
            transform=ax.transData + highlight_offset,
            zorder=5,
        )


def style_axis(ax):
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle=(0, (5, 5)), color="#7c7c7c", linewidth=0.8, alpha=0.95)
    ax.xaxis.grid(False)
    for spine in ax.spines.values():
        spine.set_color("#8a8a8a")
        spine.set_linewidth(1.8)
    ax.tick_params(axis="x", width=1.8, length=9, color="#8a8a8a")
    ax.tick_params(axis="y", width=1.6, length=8, color="#8a8a8a", labelsize=23)


def add_break_marks(ax_top, ax_bottom, size=0.012, color="#8a8a8a"):
    kwargs = dict(color=color, clip_on=False, linewidth=1.8)
    ax_top.plot((-size, +size), (-size, +size), transform=ax_top.transAxes, **kwargs)
    ax_top.plot((1 - size, 1 + size), (-size, +size), transform=ax_top.transAxes, **kwargs)
    ax_bottom.plot((-size, +size), (1 - size, 1 + size), transform=ax_bottom.transAxes, **kwargs)
    ax_bottom.plot((1 - size, 1 + size), (1 - size, 1 + size), transform=ax_bottom.transAxes, **kwargs)


def add_direction_labels(ax):
    ax.scatter([0.035], [0.11], s=260, color="#8b666f", transform=ax.transAxes, clip_on=False, zorder=6)
    ax.text(0.07, 0.105, "Risk level rising", transform=ax.transAxes, ha="left", va="center", fontsize=36)
    ax.scatter([0.49], [0.11], s=260, color="#77bed0", transform=ax.transAxes, clip_on=False, zorder=6)
    ax.text(0.525, 0.105, "Risk level declining", transform=ax.transAxes, ha="left", va="center", fontsize=36)


def plot_province_change_rate_lollipop(province_df: pd.DataFrame, output_svg: str):
    """Draw a horizontal provincial lollipop diagram and export it as SVG with editable text."""
    if "change_rate_pct" not in province_df.columns:
        if "change_rate" in province_df.columns:
            province_df = province_df.copy()
            province_df["change_rate_pct"] = pd.to_numeric(province_df["change_rate"], errors="coerce") * 100.0
        else:
            raise KeyError("CSV is missing change_rate_pct / change_rate columns.")

    plot_df = province_df[["NAME_EN_JX", "change_rate_pct"]].copy()
    plot_df["change_rate_pct"] = pd.to_numeric(plot_df["change_rate_pct"], errors="coerce")
    plot_df = plot_df.dropna(subset=["NAME_EN_JX", "change_rate_pct"]).copy()
    plot_df = plot_df.sort_values("change_rate_pct", ascending=False, kind="mergesort").reset_index(drop=True)

    if plot_df.empty:
        raise ValueError("Provincial rate of change data for plotting is empty.")

    values = plot_df["change_rate_pct"].to_numpy(dtype=float)
    y_pos = np.arange(len(plot_df))
    labels = plot_df["NAME_EN_JX"].astype(str).tolist()
    colors_local = build_point_colors(values)

    china_idx = None
    china_value = None
    china_rows = plot_df.index[plot_df["NAME_EN_JX"].str.strip().eq("China")]
    if len(china_rows) > 0:
        china_idx = int(china_rows[0])
        china_value = float(plot_df.loc[china_idx, "change_rate_pct"])

    use_break = float(np.nanmax(values)) > 50.0
    font_scale = 2.25
    fig_height = max(18.0, 0.50 * len(labels))
    fig_width = fig_height * 9.0 / 16.0

    def draw_lollipops_horizontal(ax):
        highlight_offset = mtransforms.ScaledTranslation(-2.2 / 72.0, 2.2 / 72.0, fig.dpi_scale_trans)
        for yi, xi, ci in zip(y_pos, values, colors_local):
            ax.plot(
                [0.0, xi],
                [yi, yi],
                color=ci,
                linewidth=3.6,
                alpha=0.98,
                solid_capstyle="round",
                zorder=2,
            )
            ax.scatter(
                [xi],
                [yi],
                s=235,
                color=[ci],
                edgecolor=[ci],
                linewidth=0.8,
                zorder=4,
            )
            ax.scatter(
                [xi],
                [yi],
                s=36,
                color="white",
                alpha=0.82,
                edgecolor="none",
                transform=ax.transData + highlight_offset,
                zorder=5,
            )

    def style_axis_horizontal(ax):
        ax.set_axisbelow(True)
        ax.xaxis.grid(True, linestyle=(0, (5, 5)), color="#7c7c7c", linewidth=0.8, alpha=0.95)
        ax.yaxis.grid(False)
        for spine in ax.spines.values():
            spine.set_color("#8a8a8a")
            spine.set_linewidth(1.8)
        ax.tick_params(axis="x", width=1.6, length=8, color="#8a8a8a", labelsize=23 * font_scale)
        ax.tick_params(axis="y", width=0, length=0, color="#8a8a8a", labelsize=18 * font_scale)

    def add_break_marks_horizontal(ax_left, ax_right, size=0.012, color="#8a8a8a"):
        kwargs = dict(color=color, clip_on=False, linewidth=1.8)
        ax_left.plot((1 - size, 1 + size), (-size, +size), transform=ax_left.transAxes, **kwargs)
        ax_left.plot((1 - size, 1 + size), (1 - size, 1 + size), transform=ax_left.transAxes, **kwargs)
        ax_right.plot((-size, +size), (-size, +size), transform=ax_right.transAxes, **kwargs)
        ax_right.plot((-size, +size), (1 - size, 1 + size), transform=ax_right.transAxes, **kwargs)

    if use_break:
        fig, (ax_left, ax_right) = plt.subplots(
            1,
            2,
            figsize=(fig_width, fig_height),
            sharey=True,
            gridspec_kw={"width_ratios": [4.2, 1.0], "wspace": 0.05},
        )
        for ax in (ax_left, ax_right):
            draw_lollipops_horizontal(ax)
            style_axis_horizontal(ax)

        lower_min = min(-40.0, np.floor(np.nanmin(values) / 5.0) * 5.0 - 5.0)
        lower_max = 35.0
        upper_min = 50.0
        upper_max = max(150.0, np.ceil(np.nanmax(values) / 10.0) * 10.0 + 10.0)

        ax_left.set_xlim(lower_min, lower_max)
        ax_right.set_xlim(upper_min, upper_max)
        ax_left.axvline(0, color="#0a6ea8", linewidth=3.6, zorder=3)

        left_ticks = [tick for tick in [-50, -30, -10, 0, 10, 30] if lower_min <= tick <= lower_max]
        if not left_ticks:
            left_ticks = [0.0]
        right_ticks = [tick for tick in [50.0] if upper_min <= tick <= upper_max]
        ax_left.set_xticks(left_ticks)
        ax_right.set_xticks(right_ticks)

        ax_left.spines["right"].set_visible(False)
        ax_right.spines["left"].set_visible(False)
        ax_right.tick_params(axis="y", left=False, labelleft=False)
        add_break_marks_horizontal(ax_left, ax_right)

        main_ax = ax_left
        axes_for_highlight = [ax_left, ax_right]
    else:
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        draw_lollipops_horizontal(ax)
        style_axis_horizontal(ax)
        max_abs = max(np.nanmax(np.abs(values)), 5.0)
        x_bound = np.ceil((max_abs * 1.18) / 5.0) * 5.0
        ax.set_xlim(-x_bound, x_bound)
        fixed_ticks = [tick for tick in [-50, -30, -10, 0, 10, 30, 50] if -x_bound <= tick <= x_bound]
        ax.set_xticks(fixed_ticks)
        ax.axvline(0, color="#0a6ea8", linewidth=3.6, zorder=3)

        main_ax = ax
        axes_for_highlight = [ax]

    for ax in axes_for_highlight:
        ax.set_ylim(-0.6, len(plot_df) - 0.4)
        ax.invert_yaxis()

    if china_idx is not None:
        for ax in axes_for_highlight:
            ax.axhspan(
                china_idx - 0.38,
                china_idx + 0.38,
                facecolor="none",
                edgecolor="#ec8e9c",
                linewidth=2.2,
                zorder=6,
            )

    main_ax.set_yticks(y_pos)
    main_ax.set_yticklabels(labels, rotation=0, ha="right", fontsize=18 * font_scale)
    for tick_label in main_ax.get_yticklabels():
        if tick_label.get_text().strip() == "China":
            tick_label.set_color("#d96d80")
            tick_label.set_fontweight("bold")

    fig.text(0.58, 0.02, "Raw GWR risk change rate (%)", ha="center", fontsize=28 * font_scale)

    if china_idx is not None and china_value is not None:
        direction = "increased" if china_value >= 0 else "decreased"
        annotation_text = f"Nationwide risk level\nhas {direction} by {abs(china_value):.2f}%"
        fig.text(0.985, 0.965, annotation_text, ha="right", va="top", fontsize=20 * font_scale)

    fig.subplots_adjust(left=0.34, right=0.985, bottom=0.10, top=0.96)
    fig.savefig(output_svg, bbox_inches="tight", facecolor="white", format="svg")
    print(f"SVG saved:{output_svg}")
    plt.show()


## 4. H0 / H1 / H0-H1

:
- Read point results and provincial summary results
- `H0` `H1` GeoTIFF
- Output the rate of change of `H0-H1` relative to the `H0` base GeoTIFF and map


In [ ]:
point_df, province_df = load_result_tables(point_csv, province_csv)
resolution_km = infer_resolution_km(resolution, resolution_km)

required_point_cols = [
    "Longitude",
    "Latitude",
    "H0_susceptibility_class",
    "H1_susceptibility_class",
    "climate_increment_change_rate_pct",
]
missing_point_cols = [col for col in required_point_cols if col not in point_df.columns]
if missing_point_cols:
    raise KeyError(f"Point result CSV is missing required columns: {missing_point_cols}")

if "NAME_EN_JX" not in province_df.columns:
    raise KeyError("Province summary CSV must contain NAME_EN_JX.")

boundary_shp = os.path.join(
    base_path,
    "data",
    "Administrative_divisions_of_china",
    "no_TW_AM_HK",
    "china_pro2_no_TW_AM_HK.shp",
)

h0_plot_df = point_df[["Longitude", "Latitude", "H0_susceptibility_class"]].copy()
h0_plot_df["Susceptibility_Class"] = pd.to_numeric(
    h0_plot_df["H0_susceptibility_class"],
    errors="coerce",
).round().astype("Int64")
h0_plot_df = h0_plot_df.dropna(subset=["Susceptibility_Class"]).copy()
h0_plot_df["Susceptibility_Class"] = h0_plot_df["Susceptibility_Class"].astype(int)

h1_plot_df = point_df[["Longitude", "Latitude", "H1_susceptibility_class"]].copy()
h1_plot_df["Susceptibility_Class"] = pd.to_numeric(
    h1_plot_df["H1_susceptibility_class"],
    errors="coerce",
).round().astype("Int64")
h1_plot_df = h1_plot_df.dropna(subset=["Susceptibility_Class"]).copy()
h1_plot_df["Susceptibility_Class"] = h1_plot_df["Susceptibility_Class"].astype(int)

increment_plot_df = point_df[["Longitude", "Latitude", "climate_increment_change_rate_pct"]].copy()
increment_plot_df["climate_increment_change_rate_pct"] = pd.to_numeric(
    increment_plot_df["climate_increment_change_rate_pct"],
    errors="coerce",
)
increment_plot_df = increment_plot_df.dropna(subset=["climate_increment_change_rate_pct"]).copy()

tif_h0 = os.path.join(climate_dir, f"tiff_H0_susceptibility_class_{path_name}_{ssp}_{ssp_time}.tif")
tif_h1 = os.path.join(climate_dir, f"tiff_H1_susceptibility_class_{path_name}_{ssp}_{ssp_time}.tif")
tif_increment = os.path.join(climate_dir, f"tiff_climate_increment_change_rate_pct_{path_name}_{ssp}_{ssp_time}.tif")

h0_base = os.path.join(climate_dir, f"visualization_lambert_H0_susceptibility_class_{path_name}_{ssp}_{ssp_time}")
h1_base = os.path.join(climate_dir, f"visualization_lambert_H1_susceptibility_class_{path_name}_{ssp}_{ssp_time}")
increment_base = os.path.join(climate_dir, f"visualization_lambert_climate_increment_change_rate_pct_{path_name}_{ssp}_{ssp_time}")

h0_grid, _, _, h0_bounds, _ = create_masked_raster(
    h0_plot_df,
    value_col="Susceptibility_Class",
    boundary_shp=boundary_shp,
    output_path=tif_h0,
    resolution_km=resolution_km,
    nodata=255,
    dtype="uint8",
)
create_preview_map_categorical(h0_grid, h0_bounds, h0_base, resolution_km, base_path)

h1_grid, _, _, h1_bounds, _ = create_masked_raster(
    h1_plot_df,
    value_col="Susceptibility_Class",
    boundary_shp=boundary_shp,
    output_path=tif_h1,
    resolution_km=resolution_km,
    nodata=255,
    dtype="uint8",
)
create_preview_map_categorical(h1_grid, h1_bounds, h1_base, resolution_km, base_path)

increment_grid, _, _, increment_bounds, _ = create_masked_raster(
    increment_plot_df,
    value_col="climate_increment_change_rate_pct",
    boundary_shp=boundary_shp,
    output_path=tif_increment,
    resolution_km=resolution_km,
    nodata=-9999.0,
    dtype="float32",
)
create_preview_map_continuous(
    increment_grid,
    increment_bounds,
    increment_base,
    resolution_km,
    base_path,
    colorbar_label="Risk change caused by climate change (%)",
)

print(f"H0 GeoTIFF saved: {tif_h0}")
print(f"H1 GeoTIFF saved: {tif_h1}")
print(f"Climate change rate GeoTIFF saved: {tif_increment}")


## 5. Draw provincial original GWR risk change rate map

:
- Provincial change rate lollipop chart PNG
- SVG


In [ ]:
province_svg = os.path.join(
    climate_dir,
    f"province_change_lollipop_raw_gwr_{path_name}_{ssp}_{ssp_time}.svg",
)
plot_province_change_rate_lollipop(
    province_df=province_df,
    output_svg=province_svg,
)

print(f"Province lollipop SVG saved: {province_svg}")
